In [1]:
# --- IMPORTS ---

import itertools
import os
import re
import shutil
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (ConstantKernel, Matern, WhiteKernel)
from sklearn.preprocessing import StandardScaler

In [14]:
# --- TUNING ROOM ---

CONFIG = {
    "USE_DUMMY_DATA": False,

    "PARAMETER_DEFINITIONS": {
        'roll_stiffness':       {'type': 'int', 'full_range': [4, 29]},
        'roll_stiffness_dist':  {'type': 'int', 'full_range': [-11, 14]},
        'pitch_damping':        {'type': 'int', 'full_range': [-23, 23]},
        'heave_damping':        {'type': 'int', 'full_range': [-18, 28]},
        'downforce':            {'type': 'int', 'full_range': [1, 31]},
        'cop':                  {'type': 'int', 'full_range': [-1, 55]},
    },

    "GRID_RESOLUTION": 8,  # How many points to test for each parameter in the search grid
    "LAP_GRID_RESOLUTION": 10,

    "SEARCH_EXPANSION_FACTOR": 0.5,    # 0 = training data range, 0.25 = increase by 12.5% on each side

    "SETUP_WINDOW_TOLERANCE": 1.01,     # maximum gap to the best laptime

    "MAX_WINDOWS_TO_REPORT": 3,
    "REQUIRE_PARAMETER_RANGE": False,    # so that a window is not [21, 21]

    "FALLBACK_SECTION_SENSITIVITY": 1.5,
    "FALLBACK_SECTION_MIN_SAMPLES": 5,

    "SECTION_CLUSTER_SENSITIVITY": 0.55,     # max distance between to samples, smaller means more but smaller clusters, larger means less but bigger
    "SECTION_MIN_SAMPLES": 3,

    "LAP_CLUSTER_SENSITIVITY": 0.45,
    "LAP_MIN_SAMPLES": 3,
    
    "GENERATE_CONTOUR_PLOTS": True,

    "GRID_SEARCH_CHUNK_SIZE": 50000,    # Size of chunks for the grid search to manage memory usage
}

In [15]:
# --- HELPER FUNCTIONS ---


def get_track_name(base_path: Path) -> str:

    target_dirs = [d.name for d in base_path.iterdir() if d.is_dir() and d.name.startswith('Target_')]

    if not target_dirs:
        raise FileNotFoundError(f"No 'Target_*' directory found in {base_path}")
    if len(target_dirs) > 1:
        raise ValueError(f"Multiple 'Target_*' directories found in {base_path}. Please specify one.")
    
    track_name = target_dirs[0].replace('Target_', '')
    print(f"Automatically detected target track: {track_name}")
    return track_name


def handle_output_dir(base_output_dir: Path) -> Path | None:
    """
    Manages SimXX simulation output folders. Finds the latest simulation,
    asks the user if they want to overwrite it, or creates a new one.

    Args:
        base_output_dir (Path): The root directory for all simulations (e.g., 'GPR/Real').

    Returns:
        Path | None: The full path to the chosen SimXX directory, or None if the user cancels.
    """
    base_output_dir.mkdir(parents=True, exist_ok=True)
    
    # Find all existing SimXX directories and determine the latest one
    sim_dirs = [d for d in base_output_dir.iterdir() if d.is_dir() and d.name.startswith('Sim')]
    
    latest_sim_num = 0
    if sim_dirs:
        # Extract numbers and find the max
        sim_nums = [int(d.name.replace('Sim', '')) for d in sim_dirs]
        latest_sim_num = max(sim_nums)

    if latest_sim_num > 0:
        # If simulations exist, ask the user what to do
        latest_sim_path = base_output_dir / f"Sim{latest_sim_num:02d}"
        print(f"The latest simulation found is '{latest_sim_path.name}'.")
        overwrite_choice = input("Do you want to overwrite it? (y/n): ").lower()

        if overwrite_choice in {'y', 'yes'}:
            # User wants to overwrite the latest simulation
            print(f"Preparing to overwrite '{latest_sim_path.name}'.")
            
            # Use the retry logic to safely clear the directory
            for i in range(5):
                try:
                    if latest_sim_path.exists(): shutil.rmtree(latest_sim_path)
                    break
                except (PermissionError, OSError) as e:
                    print(f"   Deletion failed: {e}. Retrying... ({i+1}/5)")
                    time.sleep(1)
            else:
                print("Error: Could not delete the directory. Please close any programs using it.")
                return None
            
            return latest_sim_path
        else:
            # User wants to create a new simulation
            new_sim_num = latest_sim_num + 1
            new_sim_path = base_output_dir / f"Sim{new_sim_num:02d}"
            print(f"Creating new simulation directory: '{new_sim_path.name}'")
            return new_sim_path
    else:
        # No simulations exist yet, create Sim01
        new_sim_path = base_output_dir / "Sim01"
        print(f"No existing simulations found. Creating '{new_sim_path.name}'.")
        return new_sim_path
        

def find_closest_value(target_value: float, value_list: list) -> float:
    return min(value_list, key=lambda x: abs(x - target_value))


def cluster_and_rank_windows(df_to_cluster, setup_parameters, time_column, config, is_fallback=False):
    """
    Performs the full pipeline of clustering, filtering, and ranking windows.
    
    Args:
        df_to_cluster (pd.DataFrame): The DataFrame of good setups to cluster.
        setup_parameters (list): List of parameter column names.
        time_column (str): The name of the column with prediction times.
        config (dict): The main CONFIG dictionary.
        is_fallback (bool): If True, use the more generous fallback settings.
        
    Returns:
        pd.DataFrame: A DataFrame of the final, ranked, and filtered optimal setups.
    """
    if df_to_cluster.empty:
        return df_to_cluster # Return empty if nothing to cluster

    # --- 1. Perform Clustering ---
    scaler = StandardScaler()
    scaled_setups = scaler.fit_transform(df_to_cluster[setup_parameters])
    
    # Choose settings based on whether this is a primary or fallback run
    if is_fallback:
        sensitivity = config["FALLBACK_SECTION_SENSITIVITY"]
        min_samples = config["FALLBACK_SECTION_MIN_SAMPLES"]
        require_range = False # Be more lenient on the fallback
        print("   -> No windows found. Retrying with generous fallback settings...")
    else:
        # This logic is for the primary run for sections OR the lap optimizer
        if 'lap' in time_column: # Check if this is the lap optimizer run
            sensitivity = config["LAP_CLUSTER_SENSITIVITY"]
            min_samples = config["LAP_MIN_SAMPLES"]
        else: # Otherwise, it's a section run
            sensitivity = config["SECTION_CLUSTER_SENSITIVITY"]
            min_samples = config["SECTION_MIN_SAMPLES"]
        require_range = config["REQUIRE_PARAMETER_RANGE"]

    dbscan = DBSCAN(eps=sensitivity, min_samples=min_samples)
    cluster_labels = dbscan.fit_predict(scaled_setups)
    df_to_cluster['window_id'] = cluster_labels
    
    clustered_df = df_to_cluster[df_to_cluster['window_id'] != -1].copy()
    if clustered_df.empty:
        return clustered_df

    # --- 2. Filter out windows that don't have a parameter range ---
    if require_range:
        valid_windows = []
        for window_id in clustered_df['window_id'].unique():
            window_data = clustered_df[clustered_df['window_id'] == window_id]
            is_valid = all(window_data[param].min() < window_data[param].max() for param in setup_parameters)
            if is_valid:
                valid_windows.append(window_id)
        clustered_df = clustered_df[clustered_df['window_id'].isin(valid_windows)]
        if clustered_df.empty:
            return clustered_df
            
    # --- 3. Rank windows by performance ---
    window_performance = clustered_df.groupby('window_id')[time_column].min().sort_values()
    rank_mapping = {old_id: rank for rank, old_id in enumerate(window_performance.index, 1)}
    clustered_df['window_id'] = clustered_df['window_id'].map(rank_mapping)
    
    # --- 4. Filter to keep only the top N windows ---
    max_windows = config["MAX_WINDOWS_TO_REPORT"]
    optimal_setups_df = clustered_df[clustered_df['window_id'] <= max_windows].copy()
    
    return optimal_setups_df

In [ ]:
# --- SECTION OPTIMISER ---

def main():
    print("--- GPR Setup Optimiser Initialising ---")

    trained_models = {}
    section_info = {}


    # --- 1. Set up Paths and Parameters ---
    data_type = 'Dummy' if CONFIG["USE_DUMMY_DATA"] else "Real"

    project_root = Path.cwd()
    
    telemetry_path = project_root/"Telemetry"/data_type

    try:
        track_name = get_track_name(telemetry_path)
    except (FileNotFoundError, ValueError) as e:
        print(f"Error: {e}")
        return
    
    track_folder_name = f"Target_{track_name.capitalize()}"

    input_file_path = telemetry_path/track_folder_name/"final_analysis_report.csv"

    base_output_dir = project_root/"GPR"/data_type

    # --- New: Get the specific SimXX path for this run ---
    active_sim_path = handle_output_dir(base_output_dir)
    if not active_sim_path:
        return None # User cancelled or deletion failed
    

    # --- 2. Load and Prepare Data ---
    print(f"\nLoading data from: {input_file_path}")
    if not input_file_path.exists():
        print(f"Error: Input file not found at {input_file_path}")
        return
    
    try:
        df = pd.read_csv(input_file_path)
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        return
    
    # Clean time delta
    if 'time_delta' in df.columns:
        df['time_delta'] = df['time_delta'].astype(str).str.replace('+', '').astype(float)

    setup_parameters = list(CONFIG["PARAMETER_DEFINITIONS"].keys())
    print(f"Analysing parameters: {', '.join(setup_parameters)}")

    full_dataset_ranges = {param: {'min': df[param].min(), 'max': df[param].max()} for param in setup_parameters}

    target_sections = sorted(df['target_track_section'].unique())
    print(f"Found {len(target_sections)} target sections to analyse: {target_sections}\n")


    # --- 3. Process Each Section ---
    for section_id in target_sections:
        print(f"--- Processing Target Section: {section_id} ---")

        
        # --- 3a. Filter and prepare data for current section ---
        section_data = df[df['target_track_section'] == section_id].copy()
        if len(section_data) < 2:
            print(f"   Warning: Skipping section {section_id} due to insufficient data ({len(section_data)} rows.)")
            continue

        # --- New: Calculate the target variable: speed in meters per second ---
        # We use training_distance and section_time from the input file.
        section_data['speed_mps'] = section_data['training_distance'] / section_data['section_time']

        X_train = section_data[setup_parameters]
        y_train = section_data['speed_mps'] # <-- Train on speed

        certainty = section_data['certainty']
        alpha = ((1.0 - certainty)**2 + 1e-6).to_numpy()

        # --- 3b. Define and train the GPR model ---
        kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, nu=1.5) + WhiteKernel(noise_level=1e-5)
        gpr = GaussianProcessRegressor(kernel=kernel, alpha=alpha, n_restarts_optimizer=10, random_state=42)
        print("Training GPR model to predict speed...")
        gpr.fit(X_train, y_train)
        print("Model training complete.")
        
        # Store the trained model and the TARGET distance for this section
        target_distance_m = section_data['target_distance'].iloc[0]
        trained_models[section_id] = gpr
        section_info[section_id] = {'target_distance_m': target_distance_m}

        # --- 3c. Perform SMART grid search and predict section time ---
        print("Performing smart grid search to predict section times...")
        
        # (The logic for building section_param_spaces is unchanged)
        section_param_spaces = {}
        for param in setup_parameters:
            param_def = CONFIG["PARAMETER_DEFINITIONS"][param]
            if CONFIG["SEARCH_EXPANSION_FACTOR"] == -1:
                search_min, search_max = param_def['full_range']
            else:
                data_min, data_max = section_data[param].min(), section_data[param].max()
                expansion = (data_max - data_min) * CONFIG["SEARCH_EXPANSION_FACTOR"]
                search_min, search_max = data_min - (expansion / 2.0), data_max + (expansion / 2.0)
            resolution = CONFIG["GRID_RESOLUTION"]
            param_grid = np.linspace(search_min, search_max, resolution)
            if param_def['type'] == 'int':
                section_param_spaces[param] = sorted(list(set(np.round(param_grid).astype(int))))
            else:
                section_param_spaces[param] = sorted(list(set(param_grid)))

        param_values_lists = [section_param_spaces[p] for p in setup_parameters]
        grid_search_iterator = itertools.product(*param_values_lists)
        total_combinations = np.prod([len(v) for v in param_values_lists])
        print(f"Total setup combinations to test: {int(total_combinations)}")
        
        chunk_size = CONFIG["GRID_SEARCH_CHUNK_SIZE"]
        all_results = []
        for i in range(0, int(total_combinations), chunk_size):
            chunk_tuples = list(itertools.islice(grid_search_iterator, chunk_size))
            if not chunk_tuples: break
            print(f"   Processing batch {i//chunk_size + 1}...")
            chunk_df = pd.DataFrame(chunk_tuples, columns=setup_parameters)
            
            # Predict speed for this chunk
            predicted_speed_mps = gpr.predict(chunk_df)
            
            # Convert predicted speed to predicted section time
            chunk_df['predicted_section_time'] = target_distance_m / predicted_speed_mps
            all_results.append(chunk_df)
            
        grid_df = pd.concat(all_results, ignore_index=True)

        # --- 3d. (Simplified) Analyze and Export Section Results ---
        print("Analyzing and exporting section results...")
        best_section_time = grid_df['predicted_section_time'].min()
        tolerance_threshold = best_section_time * CONFIG["SETUP_WINDOW_TOLERANCE"]
        good_setups_df = grid_df[grid_df['predicted_section_time'] <= tolerance_threshold].copy()
        
        # --- 3e. Cluster, Filter, and Rank Setup Windows ---
        if not good_setups_df.empty:
            
            optimal_setups_df = cluster_and_rank_windows(
                good_setups_df, setup_parameters, 'predicted_section_time', CONFIG
            )
            
            # If the primary attempt found no windows, try again with fallback settings.
            if optimal_setups_df.empty:
                optimal_setups_df = cluster_and_rank_windows(
                    good_setups_df, setup_parameters, 'predicted_section_time', CONFIG, is_fallback=True
                )

            # Define output paths
            section_id_str = f"{section_id:02d}"
            section_output_dir = active_sim_path / "Individual_Sections"
            csv_path = section_output_dir / f"section{section_id_str}_setups.csv"
            report_path = section_output_dir / f"section{section_id_str}_report.txt"
            plots_dir = section_output_dir / "Plots" / f"Section{section_id_str}"
            section_output_dir.mkdir(parents=True, exist_ok=True)
            plots_dir.mkdir(parents=True, exist_ok=True)

            # Save CSV
            optimal_setups_df = optimal_setups_df.sort_values(by=['window_id', 'predicted_section_time'])
            optimal_setups_df.to_csv(csv_path, index=False)
            print(f"   Saved optimal setups to {csv_path}")

            # Generate Text Report
            with open(report_path, 'w') as f:
                # --- New: Helper function to format a setup row into a string ---
                def format_setup(setup_series):
                    return ", ".join([f"{param}: {value}" for param, value in setup_series.items()])

                # --- New: Find the single best setup overall for this section ---
                best_overall_setup = grid_df.loc[grid_df['predicted_section_time'].idxmin()]

                f.write(f"--- Analysis for Section {section_id_str} ---\n")
                f.write(f"Fastest Predicted Section Time: {best_section_time:.3f}s\n")
                f.write(f"  - Champion Setup: {format_setup(best_overall_setup[setup_parameters])}\n\n")
                
                num_windows = optimal_setups_df['window_id'].nunique()
                f.write(f"Found {num_windows} optimal setup windows within {CONFIG['SETUP_WINDOW_TOLERANCE']:.1%} tolerance (<= {tolerance_threshold:.3f}s).\n")
                f.write("--------------------------------------------------\n\n")

                for window_id in sorted(optimal_setups_df['window_id'].unique()):
                    window_data = optimal_setups_df[optimal_setups_df['window_id'] == window_id]
                    
                    # --- New: Find the best setup within THIS window ---
                    best_setup_in_window = window_data.loc[window_data['predicted_section_time'].idxmin()]
                    best_time_in_window = best_setup_in_window['predicted_section_time']
                    delta_from_best = best_time_in_window - best_section_time
                    
                    f.write(f"Window {window_id} ({len(window_data)} setups):\n")
                    f.write(f"  - Best Time: {best_time_in_window:.3f}s ({delta_from_best:+.3f}s from section best)\n")
                    f.write(f"  - Champion Setup: {format_setup(best_setup_in_window[setup_parameters])}\n")
                    f.write("  - Parameter Ranges:\n")
                    for param in setup_parameters:
                        min_val, max_val = window_data[param].min(), window_data[param].max()
                        f.write(f"    - {param}: [{min_val}, {max_val}]\n")
                    f.write("\n") # Add a space between window reports
            print(f"   Saved analysis report to {report_path}\n")

    print("--- Section Analysis Complete ---")
    
    # (The `analysis_results` dictionary is unchanged)
    analysis_results = {
        "trained_models": trained_models,
        "section_info": section_info,
        "setup_parameters": setup_parameters,
        "full_dataset_ranges": full_dataset_ranges,
        "active_sim_path": active_sim_path
    }
    return analysis_results


if __name__ == "__main__":
    section_analysis_results = main()

--- GPR Setup Optimiser Initialising ---
Automatically detected target track: Monza
The latest simulation found is 'Sim23'.
Creating new simulation directory: 'Sim24'

Loading data from: h:\My Drive\Cranfield\11. Individual Thesis\Code\Telemetry\Real\Target_Monza\final_analysis_report.csv
Analysing parameters: roll_stiffness, roll_stiffness_dist, pitch_damping, heave_damping, downforce, cop
Found 7 target sections to analyse: [1, 2, 3, 4, 5, 6, 7]

--- Processing Target Section: 1 ---
Training GPR model to predict speed...
Model training complete.
Performing smart grid search to predict section times...
Total setup combinations to test: 143360
   Processing batch 1...


c:\Users\josea\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\josea\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


   Processing batch 2...
   Processing batch 3...
Analyzing and exporting section results...
   Saved optimal setups to h:\My Drive\Cranfield\11. Individual Thesis\Code\GPR\Real\Sim24\Individual_Sections\section01_setups.csv
   Saved analysis report to h:\My Drive\Cranfield\11. Individual Thesis\Code\GPR\Real\Sim24\Individual_Sections\section01_report.txt

--- Processing Target Section: 2 ---
Training GPR model to predict speed...


c:\Users\josea\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Model training complete.
Performing smart grid search to predict section times...
Total setup combinations to test: 172032
   Processing batch 1...
   Processing batch 2...
   Processing batch 3...
   Processing batch 4...
Analyzing and exporting section results...
   -> No windows found. Retrying with generous fallback settings...
   Saved optimal setups to h:\My Drive\Cranfield\11. Individual Thesis\Code\GPR\Real\Sim24\Individual_Sections\section02_setups.csv
   Saved analysis report to h:\My Drive\Cranfield\11. Individual Thesis\Code\GPR\Real\Sim24\Individual_Sections\section02_report.txt

--- Processing Target Section: 3 ---
Training GPR model to predict speed...


...

   Processing batch 2...
   Processing batch 3...
   Processing batch 4...
   Processing batch 5...
Analyzing and exporting section results...
   Saved optimal setups to h:\My Drive\Cranfield\11. Individual Thesis\Code\GPR\Real\Sim24\Individual_Sections\section05_setups.csv
   Saved analysis report to h:\My Drive\Cranfield\11. Individual Thesis\Code\GPR\Real\Sim24\Individual_Sections\section05_report.txt

--- Processing Target Section: 6 ---
Training GPR model to predict speed...
Model training complete.
Performing smart grid search to predict section times...
Total setup combinations to test: 143360
   Processing batch 1...


c:\Users\josea\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


   Processing batch 2...
   Processing batch 3...
Analyzing and exporting section results...
   Saved optimal setups to h:\My Drive\Cranfield\11. Individual Thesis\Code\GPR\Real\Sim24\Individual_Sections\section06_setups.csv
   Saved analysis report to h:\My Drive\Cranfield\11. Individual Thesis\Code\GPR\Real\Sim24\Individual_Sections\section06_report.txt

--- Processing Target Section: 7 ---
Training GPR model to predict speed...


c:\Users\josea\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Model training complete.
Performing smart grid search to predict section times...
Total setup combinations to test: 143360
   Processing batch 1...
   Processing batch 2...
   Processing batch 3...
Analyzing and exporting section results...
   Saved optimal setups to h:\My Drive\Cranfield\11. Individual Thesis\Code\GPR\Real\Sim24\Individual_Sections\section07_setups.csv
   Saved analysis report to h:\My Drive\Cranfield\11. Individual Thesis\Code\GPR\Real\Sim24\Individual_Sections\section07_report.txt

--- Section Analysis Complete ---


In [17]:
# --- LAP OPTIMISER FUNCTION ---


def lap_optimiser(analysis_results, config):


    # --- 1. Unpack Analysis Results ---

    trained_models = analysis_results["trained_models"]
    section_info = analysis_results["section_info"]
    setup_parameters = analysis_results["setup_parameters"]
    full_dataset_ranges = analysis_results["full_dataset_ranges"]
    active_sim_path = analysis_results["active_sim_path"]

    # Define output paths
    data_type = "Dummy" if config["USE_DUMMY_DATA"] else "Real"
    project_root = Path.cwd()
    output_base_path = project_root / "GPR" / data_type


    # --- 2. Build the Global Smart Grid ---

    print("Building global smart grid for lap optimisation...")

    global_param_spaces = {}
    for param in setup_parameters:
        param_def = config["PARAMETER_DEFINITIONS"][param]

        if config["SEARCH_EXPANSION_FACTOR"] == -1:
            search_min, search_max = param_def['full_range']
        else:
            data_min = full_dataset_ranges[param]['min']
            data_max = full_dataset_ranges[param]['max']
            data_range = data_max - data_min

            expansion = data_range * config["SEARCH_EXPANSION_FACTOR"]
            search_min = data_min - (expansion / 2.0)
            search_max = data_max + (expansion / 2.0)

            search_min = max(search_min, param_def['full_range'][0])
            search_max = min(search_max, param_def['full_range'][1])

        if param_def['type'] == 'int':
            search_min = int(np.ceil(search_min))
            search_max = int(np.floor(search_max))

        resolution = config["LAP_GRID_RESOLUTION"]

        # Use a set of unique values in case range is zero
        param_grid = np.linspace(search_min, search_max, resolution)

        if param_def['type'] == 'int':
            global_param_spaces[param] = sorted(list(set(np.round(param_grid).astype(int))))
        else:
            global_param_spaces[param] = sorted(list(set(param_grid)))

    # Creates list of all possible values for each parameter
    param_values_lists = [global_param_spaces[p] for p in setup_parameters]

    
    # --- 3. Perform Grid Search for the Whole Lap ---
    
    # Generates the Cartesian product of all parameters to create a grid
    grid_search_iterator = itertools.product(*param_values_lists)
    total_combinations = np.prod([len(v) for v in param_values_lists])
    print(f"Total setup combinations to test for the lap: {int(total_combinations)}")
    
    chunk_size = config["GRID_SEARCH_CHUNK_SIZE"]
    all_lap_results = []

    for i in range(0, int(total_combinations), chunk_size):
        chunk_tuples = list(itertools.islice(grid_search_iterator, chunk_size))
        if not chunk_tuples: break
        print(f"   Processing lap batch {i//chunk_size + 1}/{int(np.ceil(total_combinations/chunk_size))}...")
        chunk_df = pd.DataFrame(chunk_tuples, columns=setup_parameters)
        
        # This will hold the summed lap time for each setup in the chunk
        total_lap_time = np.zeros(len(chunk_df))
        
        # Loop through each section model, predict speed, and calculate section time
        for section_id, model in trained_models.items():
            dist_m = section_info[section_id]['target_distance_m']
            predicted_speed_mps = model.predict(chunk_df[setup_parameters])
            
            # Add this section's time (distance / speed) to the running total for the lap
            total_lap_time += dist_m / predicted_speed_mps
            
        chunk_df['predicted_lap_time'] = total_lap_time
        all_lap_results.append(chunk_df)

    print("Combining lap batch results...")
    lap_grid_df = pd.concat(all_lap_results, ignore_index=True)

    # --- 4. Analyse Lap Results and Identify Windows ---
    print("Analyzing lap results to identify optimal windows...")
    
    # --- New: Find the single setup that produces the fastest realistic lap time ---
    fastest_lap_time = lap_grid_df['predicted_lap_time'].min()
    tolerance_threshold = fastest_lap_time * config["SETUP_WINDOW_TOLERANCE"]

    good_lap_setups_df = lap_grid_df[lap_grid_df['predicted_lap_time'] <= tolerance_threshold].copy()
    print(f"Found {len(good_lap_setups_df)} total lap setups within {config['SETUP_WINDOW_TOLERANCE']:.1%} of the best lap time ({fastest_lap_time:.3f}s).")
    
    # --- 5. Cluster and Export Lap Results ---
    if not good_lap_setups_df.empty:
        # --- New: Use the helper function for a cleaner workflow ---
        optimal_lap_setups_df = cluster_and_rank_windows(
            good_lap_setups_df, setup_parameters, 'predicted_lap_time', config
        )

        # Filter out noise points
        clustered_df = good_lap_setups_df[good_lap_setups_df['window_id'] != -1].copy()

        # --- New: Filter out windows that don't have a parameter range ---
        if config["REQUIRE_PARAMETER_RANGE"] and not clustered_df.empty:
            valid_windows = []
            for window_id in clustered_df['window_id'].unique():
                window_data = clustered_df[clustered_df['window_id'] == window_id]
                is_valid = True
                for param in setup_parameters:
                    if window_data[param].min() == window_data[param].max():
                        is_valid = False
                        break
                if is_valid:
                    valid_windows.append(window_id)
            clustered_df = clustered_df[clustered_df['window_id'].isin(valid_windows)]

        # --- New: Rank windows by performance ---
        if not clustered_df.empty:
            # Find the best time for each window
            window_performance = clustered_df.groupby('window_id')['predicted_lap_time'].min().sort_values()
            
            # Create a mapping from old ID to new rank (1st, 2nd, 3rd...)
            rank_mapping = {old_id: rank for rank, old_id in enumerate(window_performance.index, 1)}
            clustered_df['window_id'] = clustered_df['window_id'].map(rank_mapping)
            
            # --- New: Filter to keep only the top N windows ---
            max_windows = config["MAX_WINDOWS_TO_REPORT"]
            optimal_lap_setups_df = clustered_df[clustered_df['window_id'] <= max_windows].copy()
        else:
            optimal_lap_setups_df = clustered_df # Use empty df if all were filtered
        # Define file paths
        lap_output_dir = active_sim_path / "Lap_Optimization"
        lap_output_dir.mkdir(parents=True, exist_ok=True)
        csv_path = lap_output_dir / "lap_optimal_setups.csv"
        report_path = lap_output_dir / "lap_report.txt"

        # Sort and save CSV
        optimal_lap_setups_df = optimal_lap_setups_df.sort_values(by=['window_id', 'predicted_lap_time'])
        optimal_lap_setups_df.to_csv(csv_path, index=False)
        print(f"   Saved optimal lap setups to {csv_path}")

        # --- New: Generate the final, realistic lap time report ---
        with open(report_path, 'w') as f:
            # Helper function to format seconds into M:SS.TTT
            def format_time(seconds):
                minutes = int(seconds // 60)
                sec = int(seconds % 60)
                thousandths = int((seconds * 1000) % 1000)
                return f"{minutes}:{sec:02d}.{thousandths:03d}"
            
            # Helper function to format a setup row into a string
            def format_setup(setup_series):
                return ", ".join([f"{param}: {value}" for param, value in setup_series.items()])

            # --- New: Find the single best setup overall for the entire lap ---
            best_overall_setup = lap_grid_df.loc[lap_grid_df['predicted_lap_time'].idxmin()]
            
            f.write("--- Analysis for Entire Lap ---\n")
            f.write(f"Fastest Predicted Lap Time: {format_time(fastest_lap_time)}\n")
            f.write(f"  - Champion Setup: {format_setup(best_overall_setup[setup_parameters])}\n\n")
            
            num_windows = optimal_lap_setups_df['window_id'].nunique()
            f.write(f"Found {num_windows} optimal setup windows within {config['SETUP_WINDOW_TOLERANCE']:.1%} tolerance (slower than {format_time(tolerance_threshold)}).\n")
            f.write("--------------------------------------------------\n\n")

            for window_id in sorted(optimal_lap_setups_df['window_id'].unique()):
                window_data = optimal_lap_setups_df[optimal_lap_setups_df['window_id'] == window_id]
                
                # --- New: Find the best setup within THIS window ---
                best_setup_in_window = window_data.loc[window_data['predicted_lap_time'].idxmin()]
                best_time_in_window = best_setup_in_window['predicted_lap_time']
                delta_from_best = best_time_in_window - fastest_lap_time
                percent_from_best = (delta_from_best / fastest_lap_time) * 100 if fastest_lap_time > 0 else 0

                f.write(f"Window {window_id} ({len(window_data)} setups):\n")
                f.write(f"  - Best Time: {format_time(best_time_in_window)} ({delta_from_best:+.3f}s / +{percent_from_best:.2f}% from lap best)\n")
                f.write(f"  - Champion Setup: {format_setup(best_setup_in_window[setup_parameters])}\n")
                f.write("  - Parameter Ranges:\n")
                for param in setup_parameters:
                    min_val, max_val = window_data[param].min(), window_data[param].max()
                    f.write(f"    - {param}: [{min_val}, {max_val}]\n")
                f.write("\n") # Add a space between window reports
        
        print(f"   Saved lap analysis report to {report_path}\n")

    else:
        print("No optimal lap setup windows found within the specified tolerance.\n")
    
    print("--- Lap Optimisation Complete ---")


if 'section_analysis_results' in locals() and section_analysis_results is not None:
    lap_optimiser(section_analysis_results, CONFIG)
else:
    print("Error: Please run the main analysis cell above to generate the section models before running the lap optimiser.")

Building global smart grid for lap optimisation...
Total setup combinations to test for the lap: 900000
   Processing lap batch 1/18...
   Processing lap batch 2/18...
   Processing lap batch 3/18...
   Processing lap batch 4/18...
   Processing lap batch 5/18...
   Processing lap batch 6/18...
   Processing lap batch 7/18...
   Processing lap batch 8/18...
   Processing lap batch 9/18...
   Processing lap batch 10/18...
   Processing lap batch 11/18...
   Processing lap batch 12/18...
   Processing lap batch 13/18...
   Processing lap batch 14/18...
   Processing lap batch 15/18...
   Processing lap batch 16/18...
   Processing lap batch 17/18...
   Processing lap batch 18/18...
Combining lap batch results...
Analyzing lap results to identify optimal windows...
Found 46124 total lap setups within 101.0% of the best lap time (112.804s).
   Saved optimal lap setups to h:\My Drive\Cranfield\11. Individual Thesis\Code\GPR\Real\Sim24\Lap_Optimization\lap_optimal_setups.csv
   Saved lap ana